In [2]:
import torch
import numpy as np
import torch.nn as nn

#----PyTorch Tutorial 5 [Gradient Descent with Autograd & Backpropagation]-------

In [ ]:
# For Linear Regression (using only numpy)
# Manual Prediction, Gradient Computation, Loss Computation and Parameter updates
# f = w * x, no bias

# f = 2 * x
x = np.array([1,2,3,4], dtype=np.float32)
y = np.array([2,4,6,8], dtype=np.float32)

w = 0.0

# model prediction
def forward(x):
    return w * x

# loss = MSE
def loss(y, y_predicted):
    return ((y_predicted - y)**2).mean()

# gradient
# MSE = 1/N * (w*x - y)**2
# Derivative of above: dJ/dw = 1/N 2x (w*x - y)
def gradient(x, y, y_predicted):
    return np.dot(2*x, (y_predicted - y)).mean()

print(f'Prediction before training: f(5) = {forward(5):.3f}')

# Training
learning_rate = 0.01
n_iters = 10

for epoch in range(n_iters):
    # prediction = forward pass
    y_pred = forward(x)

    #loss
    l = loss(y, y_pred)

    #gradients
    dw = gradient(x,y,y_pred) # negative value
    
    # update weights (the optimizer)
    w -= learning_rate *dw   # w - (-dw)

    if epoch % 1 == 0:
        print(f'epoch{epoch+1}: w = {w:.3f}, loss = {l:.8f}')

print(f'prediction after training: f(5) = {forward(5):.3f}')


In [5]:
# For Linear Regression (using Pytorch)
# Gradient Computation done by Autograd
# f = w * x, no bias

# f = 2 * x
x = torch.tensor([1,2,3,4], dtype=torch.float32)
y = torch.tensor([2,4,6,8], dtype=torch.float32)

w = torch.tensor(0.0, dtype = torch.float32, requires_grad=True)

# model prediction
def forward(x):
    return w * x

# loss = MSE
def loss(y, y_predicted):
    return ((y_predicted - y)**2).mean()

# No gradient function

print(f'Prediction before training: f(5) = {forward(5):.3f}')

# Training
learning_rate = 0.01
n_iters = 20

for epoch in range(n_iters):
    # prediction = forward pass
    y_pred = forward(x)

    #loss
    l = loss(y, y_pred)

    #gradient = backward pass  
    l.backward() # dl/dw
    
    # update weights
    with torch.no_grad():
        w -= learning_rate *w.grad   # w - (-dw)
    
    #zero.gradients
    w.grad.zero_()

    if epoch % 10 == 0:
        print(f'epoch{epoch+1}: w = {w:.3f}, loss = {l:.8f}')

print(f'prediction after training: f(5) = {forward(5):.3f}')


Prediction before training: f(5) = 0.000
epoch1: w = 0.300, loss = 30.00000000
epoch11: w = 1.665, loss = 1.16278565
prediction after training: f(5) = 9.612


In [6]:
# For Linear Regression (using Pytorch)
# Prediction: manual, Gradient Computation: Autograd, 
# Loss Computation: Pytorch Loss, Parameter updates: Pytorch Optimizer

# 1) Design model (input, output size, forward pass)
# 2) Construct loss and optimizer
# 3) Training loop
#    - forward pass: compute prediction
#    - backward pass: gradients
#    - update weights

# f = w * x, no bias

# f = 2 * x
x = torch.tensor([1,2,3,4], dtype=torch.float32)
y = torch.tensor([2,4,6,8], dtype=torch.float32)

w = torch.tensor(0.0, dtype = torch.float32, requires_grad=True)

# model prediction
def forward(x):
    return w * x

# No manual loss function
# No manual gradient function

print(f'Prediction before training: f(5) = {forward(5):.3f}')

# Training
learning_rate = 0.01
n_iters = 20

loss = nn.MSELoss()
optimizer = torch.optim.SGD([w], lr = learning_rate)

for epoch in range(n_iters):
    # prediction = forward pass
    y_pred = forward(x)

    #loss
    l = loss(y, y_pred)

    #gradient = backward pass  
    l.backward() # dl/dw
    
    # update weights
    optimizer.step()
    
    #zero.gradients
    optimizer.zero_grad()

    if epoch % 2 == 0:
        print(f'epoch{epoch+1}: w = {w:.3f}, loss = {l:.8f}')

print(f'prediction after training: f(5) = {forward(5):.3f}')

Prediction before training: f(5) = 0.000
epoch1: w = 0.300, loss = 30.00000000
epoch3: w = 0.772, loss = 15.66018772
epoch5: w = 1.113, loss = 8.17471695
epoch7: w = 1.359, loss = 4.26725292
epoch9: w = 1.537, loss = 2.22753215
epoch11: w = 1.665, loss = 1.16278565
epoch13: w = 1.758, loss = 0.60698116
epoch15: w = 1.825, loss = 0.31684780
epoch17: w = 1.874, loss = 0.16539653
epoch19: w = 1.909, loss = 0.08633806
prediction after training: f(5) = 9.612


In [3]:
# For Linear Regression (using Pytorch)
# Prediction: Pytorch model, Gradient Computation: Autograd, 
# Loss Computation: Pytorch Loss, Parameter updates: Pytorch Optimizer

# 1) Design model (input, output size, forward pass)
# 2) Construct loss and optimizer
# 3) Training loop
#    - forward pass: compute prediction
#    - backward pass: gradients
#    - update weights

# f = w * x, no bias

# f = 2 * x
x = torch.tensor([[1],[2],[3],[4]], dtype=torch.float32)
y = torch.tensor([[2],[4],[6],[8]], dtype=torch.float32)

X_test = torch.tensor([5], dtype=torch.float32)

n_samples, n_features = x.shape
print(n_samples, n_features)

input_size = n_features
output_size = n_features

# no manual input of weight

# model prediction
#model = nn.Linear(input_size, output_size) # One layer

class LinearRegression(nn.Module): # inheritance, class is extending PyTorch,s base nn.Module class
# Think of nn.module as a blueprint that every pytorch model must follow, getting a ton of functionality for free

    def __init__(self, input_dim, output_dim): # Constructor, input_size and output_size go here, job is to define and register all layers
        super(LinearRegression, self).__init__()
        # super(): Refers to parent class (nn.Module)
        # LinearRegression, self: tells python which class and instance we,re in
        # init(): Calls parent constructor
        # Reason? When inheriting from nn.Module, its own __init__ sets up internal machinery. 
        # If we dont call super().__init_() first, that machinery never gets initialized and everything breaks
        
        # define layers
        self.lin = nn.Linear(input_dim, output_dim)
        # Self: attaches the layer to this specific model instance
        #.lin: name of variable
        # nn.linear: PyTorch built in linear layer, internally does: (y = mx+b)
    
    def forward(self,x):
        return self.lin(x)

model = LinearRegression(input_size, output_size)
# No manual loss function
# No manual gradient function
print(f'Prediction before training: f(5) = {model(X_test).item():.3f}')

# Training
learning_rate = 0.01
n_iters = 1000

loss = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr = learning_rate)

for epoch in range(n_iters):
    # prediction = forward pass
    y_pred = model(x)

    #loss
    l = loss(y, y_pred)

    #gradient = backward pass  
    l.backward() # dl/dw
    
    # update weights
    optimizer.step()
    
    #zero.gradients
    optimizer.zero_grad()

    if epoch % 100 == 0:
        [w,b] = model.parameters()
        print(f'epoch{epoch+1}: w = {w[0][0].item():.3f}, loss = {l:.8f}')

print(f'prediction after training: f(5) = {model(X_test).item():.3f}')

4 1
Prediction before training: f(5) = -2.706
epoch1: w = -0.144, loss = 48.79661560
epoch101: w = 1.822, loss = 0.04581838
epoch201: w = 1.868, loss = 0.02515377
epoch301: w = 1.902, loss = 0.01380909
epoch401: w = 1.928, loss = 0.00758104
epoch501: w = 1.946, loss = 0.00416189
epoch601: w = 1.960, loss = 0.00228484
epoch701: w = 1.971, loss = 0.00125434
epoch801: w = 1.978, loss = 0.00068862
epoch901: w = 1.984, loss = 0.00037805
prediction after training: f(5) = 9.975
